In [ ]:
import re
import os
import random
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select

import pandas as pd
import numpy as np

#1° ETAPA (DISPARAR CLIENTES NA LISTA);
#2° ETAPA (ATENDER CLIENTES NA FILA AUTOMATICAMENTE)
#3° ETAPA (CAPTURAR INDICADORES DOS ATENDIMENTOS DURANTE A AUTOMAÇÃO)


In [70]:
def get_driver(browser=None):

    if browser == "chrome":
        return webdriver.Chrome()

    elif browser == "firefox":
        return webdriver.Firefox()

    elif browser == "edge":
        return webdriver.Edge()

    elif browser == "brave":
        options = Options()
        options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
        return webdriver.Chrome(options=options)

    else:
        raise ValueError("Sem driver no selenium")


driver = get_driver("brave")  
driver.get("https://prd-sac.onrender.com/sistema/lista_atendimentos.php")
wait = WebDriverWait(driver, 20)

In [ ]:
def iniciar_atendimento():

    print("Atendimento CHAT - Iniciado")
    for i in range(2):      

        select = wait.until(EC.presence_of_element_located((By.NAME, 'resposta_atendente')))  
        select_chat = Select(select)
    
        if select_chat.options[1:]:
            opcao = random.choice(select_chat.options[1:])     
            opcao.click()

        botao = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/div[1]/form/button')))
        botao.click()
        

In [ ]:
def extrair_protocolos(driver):
    df_protocolos = []

    tr = wait.until(EC.presence_of_all_elements_located((By.XPATH, '//table/tbody/tr')))
    trs = len(tr)

    for i in range(1, trs + 1):
        col = driver.find_elements(By.XPATH, f'//table/tbody/tr[{i}]/td')

        if len(col) >= 8:
            status_text = col[6].text.strip()

            df_protocolos.append({
                "protocolo": col[0].text,
                "servico": col[2].text,
                "validade": col[4].text,
                "status": col[5].text,
                "feedback": col[6].text
            })

            if status_text == "-":
                botao = wait.until(EC.element_to_be_clickable((By.XPATH, f'/html/body/div/table/tbody/tr[{i}]/td[8]/a/button')))   
                aba_lista = driver.current_window_handle

                simular_atraso_atendentente = random.uniform(5, 15)
                print(f"Aguardando atendimento ser inicializado.")
                time.sleep(simular_atraso_atendentente)
                
                botao.click()  
                driver.switch_to.window(driver.window_handles[-1])
                iniciar_atendimento()
                print("Atendimento CHAT - Finalizado")
                wait.until(EC.presence_of_all_elements_located((By.XPATH, '//table/tbody/tr')))
                

    df = pd.DataFrame(df_protocolos)
    print(df)
    return df

extrair_protocolos(driver)

Atendimento CHAT - Iniciado
Atendimento CHAT - Finalizado
Atendimento CHAT - Iniciado
Atendimento CHAT - Finalizado


TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff629c11035+5a95]
	chromedriver!(No symbol) [0x7ff629b32c70]
	chromedriver!(No symbol) [0x7ff629954a7d]
	chromedriver!(No symbol) [0x7ff6299aff91]
	chromedriver!(No symbol) [0x7ff6299b025c]
	chromedriver!(No symbol) [0x7ff629a02f67]
	chromedriver!(No symbol) [0x7ff6299ffbea]
	chromedriver!(No symbol) [0x7ff6299a1d02]
	chromedriver!(No symbol) [0x7ff6299a2b13]
	chromedriver!GetHandleVerifier [0x7ff62a031411+425e71]
	chromedriver!GetHandleVerifier [0x7ff62a05ca47+4514a7]
	chromedriver!GetHandleVerifier [0x7ff62a05094e+4453ae]
	chromedriver!GetHandleVerifier [0x7ff629d0bf6e+1009ce]
	chromedriver!(No symbol) [0x7ff629b40a05]
	chromedriver!(No symbol) [0x7ff629b3bcf4]
	chromedriver!(No symbol) [0x7ff629b3be84]
	chromedriver!(No symbol) [0x7ff629b266ac]
	KERNEL32!BaseThreadInitThunk [0x7ffb184f7374+14]
	ntdll!RtlUserThreadStart [0x7ffb19a5cc91+21]
